In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
%matplotlib inline

# Load processed data
df = pd.read_csv('../data/raw/reviews_with_sentiment.csv')
print(f"✅ Data loaded! Shape: {df.shape}")
df.head(3)

In [ ]:
# ============================================
# IDENTIFY DRIVERS AND PAIN POINTS PER BANK
# ============================================

banks = df['bank'].unique()

for bank in banks:
    bank_df = df[df['bank'] == bank]
    
    print(f"\n{'='*50}")
    print(f"BANK: {bank}")
    print(f"{'='*50}")
    
    # Satisfaction drivers — positive reviews
    pos = bank_df[bank_df['sentiment_label'] == 'positive']
    print(f"\n✅ SATISFACTION DRIVERS (from {len(pos)} positive reviews):")
    
    # Top themes in positive reviews
    pos_themes = pos['identified_theme'].value_counts().head(3)
    for theme, count in pos_themes.items():
        pct = count/len(pos)*100
        print(f"  → {theme}: {count} reviews ({pct:.1f}%)")
    
    # Sample positive reviews
    print(f"\n  Sample positive reviews:")
    for rev in pos['review_text'].head(3).values:
        print(f"    \"{str(rev)[:80]}...\"")
    
    # Pain points — negative reviews
    neg = bank_df[bank_df['sentiment_label'] == 'negative']
    print(f"\n❌ PAIN POINTS (from {len(neg)} negative reviews):")
    
    # Top themes in negative reviews
    neg_themes = neg['identified_theme'].value_counts().head(3)
    for theme, count in neg_themes.items():
        pct = count/len(neg)*100
        print(f"  → {theme}: {count} reviews ({pct:.1f}%)")
    
    # Sample negative reviews
    print(f"\n  Sample negative reviews:")
    for rev in neg['review_text'].head(3).values:
        print(f"    \"{str(rev)[:80]}...\"")

In [ ]:
# Compare banks across key dimensions
print("=== BANK COMPARISON DASHBOARD ===\n")

comparison = []
for bank in banks:
    bank_df = df[df['bank'] == bank]
    comparison.append({
        'Bank': bank,
        'Total Reviews': len(bank_df),
        'Avg Rating': round(bank_df['rating'].mean(), 2),
        'Positive %': round((bank_df['sentiment_label']=='positive').mean()*100, 1),
        'Negative %': round((bank_df['sentiment_label']=='negative').mean()*100, 1),
        'Avg Sentiment': round(bank_df['sentiment_score'].mean(), 4),
        'Top Theme': bank_df['identified_theme'].mode()[0]
    })

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

In [ ]:
print("""
=== BANK-SPECIFIC PRODUCT RECOMMENDATIONS ===

COMMERCIAL BANK OF ETHIOPIA (CBE):
  Satisfaction Drivers:
  → Large user base trusts the brand (456 reviews, established presence)
  → Core transfer functionality works for majority of users
  
  Pain Points & Recommendations:
  1. TRANSACTION SPEED: Fix slow loading during transfers
     → Recommendation: Implement background processing and 
       progress indicators to reduce perceived wait time
  2. APP STABILITY: Address crashes on older Android versions
     → Recommendation: Expand device testing matrix to include
       Android 8+ devices common in Ethiopian market

BANK OF ABYSSINIA (BOA):
  Satisfaction Drivers:
  → Highest review volume (498) shows strong user engagement
  → UI design receives positive mentions
  
  Pain Points & Recommendations:
  1. LOGIN ISSUES: OTP delivery failures reported frequently
     → Recommendation: Add backup authentication (fingerprint/PIN)
       and improve Ethio Telecom SMS gateway reliability
  2. FEATURE GAPS: Users request budgeting and spending analytics
     → Recommendation: Prioritize mini-statement and spending 
       category features in next sprint

DASHEN BANK:
  Satisfaction Drivers:
  → Highest 5-star sentiment score (0.4456) — most satisfied users
     are very expressive in their praise
  → Super App approach appreciated by tech-savvy users
  
  Pain Points & Recommendations:
  1. ONBOARDING: New users struggle with account activation
     → Recommendation: Add in-app tutorial and live chat support
       for first-time users
  2. TRANSFER LIMITS: Users request higher daily limits
     → Recommendation: Implement tiered limits based on KYC 
       verification level with clear upgrade path

CROSS-BANK RECOMMENDATION:
  → All three banks should implement an AI chatbot for common 
    queries (balance check, transfer status, OTP issues) to 
    reduce support load and improve response time 24/7.
""")

In [ ]:
# Heatmap of avg sentiment by bank and theme
pivot = df.groupby(['bank', 'identified_theme'])['sentiment_score'].mean().unstack(fill_value=0)

plt.figure(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'label': 'Avg Sentiment Score'})
plt.title('Average Sentiment Score by Bank and Theme', fontsize=14, fontweight='bold')
plt.xlabel('Theme')
plt.ylabel('Bank')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/raw/sentiment_heatmap.png', dpi=150)
plt.show()
print("✅ Heatmap saved!")

In [ ]:
colors_map = {
    'Commercial Bank of Ethiopia': 'steelblue',
    'Bank of Abyssinia': 'coral',
    'Dashen Bank': 'green'
}

plt.figure(figsize=(10, 6))
for bank in banks:
    bank_df = df[df['bank'] == bank]
    plt.scatter(bank_df['rating'], bank_df['sentiment_score'],
                alpha=0.3, label=bank,
                color=colors_map[bank], s=30)

plt.title('Rating vs Sentiment Score by Bank', fontsize=14, fontweight='bold')
plt.xlabel('Star Rating', fontsize=12)
plt.ylabel('Sentiment Score', fontsize=12)
plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/rating_vs_sentiment.png', dpi=150)
plt.show()
print("✅ Scatter plot saved!")

In [ ]:
# Final save
df.to_csv('../data/raw/final_analysis.csv', index=False)
print(f"✅ Final dataset saved!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print("\n=== TASK 4 COMPLETE ===")